# nb05: Why is TimesFM "shit"? — diagnosis + predictions-vs-actuals (DIAGNOSTIC ONLY)

* * *

**QUESTION:** *Why* did RAW TimesFM-2.0-500m (close channel) score future-volatility val R² = **−0.134** in nb04 — below the untrained-random floor (+0.097) and far below the 24-d moment baseline (+0.333)? Can we *see* the failure by plotting predictions against actuals, and which single lever (per-sample normalization, channel count, pooling, or probe regularization) is responsible?

**H1:** The failure is an INPUT/USAGE artifact, dominated by per-sample causal normalization erasing the absolute volatility level. Predicted-vs-actual plots will show TimesFM (and the normalized encoders) as a structureless cloud while a raw-σ feature on UN-normalized context recovers most of the +0.333 — i.e. the input pipeline, not the backbone, is the lever.

**H0 (null):** TimesFM features genuinely carry no linearly-decodable future-volatility signal regardless of normalization/channel/pooling — un-normalizing, all-channels, last-pool, and a wider α grid all leave it at/below the random floor, and its forecast head also fails to track the series.

**MEASURED TARGETS** (one harness, one seed, one WIDE α grid for all):
- **Test 1 — output shape:** real `last_hidden_state.shape` / n_positions for our L=64 input (settles whether mean-pool averages pad positions).
- **Test 2 — forecast vs actual:** TimesFM forecast-head `mean_predictions` overlaid on the true H-step target, ~9 val windows (normalized close log-returns).
- **Test 3 — predicted vs actual:** ridge-probe predictions vs true future_volatility for raw-input / untrained z_price / trained z_price / TimesFM, with val R² annotated.
- **Test 4 — fix matrix (forward-passes only):** future_volatility val R² for {raw-input, raw close-σ, TimesFM} × {normalized vs raw context} + {mean vs last pool}, all WIDE grid.

**DECISION RULE.**
- If `raw close-σ (RAW ctx)` val R² ≳ +0.30 (≈ the +0.333 baseline) while `raw close-σ (NORMALIZED ctx)` ≈ 0 → **per-sample normalization removes the volatility-level signal; the INPUT PIPELINE is the lever.** Next: feed un-normalized context (or an explicit raw-σ channel) to any backbone.
- If un-normalizing the context lifts `TimesFM (RAW ctx)` materially above its normalized score AND above the floor → TimesFM *can* carry vol once it sees scale; a frozen-TimesFM pivot becomes worth re-evaluating (but only after the pipeline fix).
- If NOTHING (un-norm, last-pool) lifts TimesFM above the floor → H0: TimesFM close-channel is genuinely uninformative here; do not pivot.
- Test 1: if n_positions ≫ ~2, mean-pool is corrupted by padding → re-read Test 3/4 TimesFM numbers under `pool="last"`.

**PRIORS / ASSUMPTIONS:**
- nb04 verdict stands: HOLD on the TimesFM pivot. This notebook is a *post-mortem*, forward-passes only, NO training units.
- Probe targets are derived from RAW (unnormalized) target windows; only the ENCODER INPUT normalization is toggled by `normalize_context`.
- TimesFM via the HF `transformers` integration (`TimesFmModelForPrediction`), NOT the `timesfm` pip package (numpy<2 risk). `last_hidden_state` for the probe; `mean_predictions` for the forecast overlay — both verified empirically in Test 1 before being trusted.
- The 24-d moment baseline (+0.333) is the bar to beat, not the +0.097 random floor.

**FALSIFIER (of H1):** raw close-σ on UN-normalized context does NOT recover the volatility signal (val R² ≈ 0, far below +0.33), i.e. the normalization is not what's removing it — then the input-pipeline hypothesis is wrong and the signal lives somewhere normalization doesn't touch.

**RESULT** (Colab; fill after running): _pending Colab run_

**DECISION + NEXT ACTION** (fill after running): _pending Colab run_

In [ ]:
# == Colab setup (skipped in VS Code / local kernel) ==
import os, sys

IN_COLAB = 'google.colab' in sys.modules
IN_VSCODE = 'VSCODE_PID' in os.environ or 'VSCODE_CWD' in os.environ
if IN_VSCODE:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd /content
    !git clone https://github.com/shreyasnat2804/JEPA-quant.git 2>/dev/null || (cd JEPA-quant && git pull)
    %cd /content/JEPA-quant
    # transformers>=4.48 is REQUIRED for the TimesFM integration (TimesFmModelForPrediction).
    # We use the HF integration, NOT the `timesfm` pip package (it can pin numpy<2 and
    # break Colab's numpy-2 stack — same trap as Moirai/uni2ts).
    %pip install -q "transformers>=4.48" "peft>=0.11" accelerate einops matplotlib pyarrow
    # Colab preinstalls torchao 0.10.0; recent PEFT raises (not returns False) below its
    # 0.16.0 minimum, blowing up get_peft_model(). Remove it (we don't quantize).
    %pip uninstall -q -y torchao

# Add src to path regardless of environment
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..' if 'notebooks' in os.getcwd() else '.'))
src_path = os.path.join(repo_root, 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)
print('repo_root:', repo_root)
print('IN_COLAB:', IN_COLAB, '  IN_VSCODE:', IN_VSCODE)

In [ ]:
# == autoreload shim (imp removed in Python 3.12) ==
import types, importlib
if 'imp' not in sys.modules:
    _imp_shim = types.ModuleType('imp')
    _imp_shim.reload = importlib.reload
    sys.modules['imp'] = _imp_shim
%load_ext autoreload
%autoreload 2

In [ ]:
# == user configuration: edit before running ==
import torch

# nb03 best checkpoint (same as nb03c / nb04).
CHECKPOINT_PATH = '/content/drive/MyDrive/Colab Notebooks/JEPA-QUANT/checkpoints/nb03_best.pt'
# Per-ticker .parquet directory (same as training / nb04).
DATA_DIR = '/content/drive/MyDrive/Colab Notebooks/JEPA-QUANT/data/raw/stocks'
# HF TimesFM checkpoint (PyTorch). 2.0-500m: patch_length=32, hidden_size=1280, 50 layers.
TIMESFM_CHECKPOINT = 'google/timesfm-2.0-500m-pytorch'

N_TRAIN_BATCHES = 100   # 100 * 256 = 25,600 samples nominal (matches nb04)
N_VAL_BATCHES = 50      # 50  * 256 = 12,800 samples nominal
N_FORECAST_SAMPLES = 9  # forecast-vs-actual overlay panes
HORIZON = 16            # H steps to keep from TimesFM's forecast head (= data.horizon)

# WIDE ridge sweep (same as nb04). Test 4 widens further if anything pins on a boundary.
WIDE_ALPHAS = (0.01, 0.1, 1.0, 10.0, 100.0, 1000.0, 10000.0)
SEED = 42
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

In [ ]:
# == build config, load trained checkpoint, build untrained baseline ==
# Architecture matches nb03 / nb03c / nb04: d_model=768, n_layers=8, n_heads=12,
# freeze_backbone=True. (Only the z_price encoders use this; TimesFM/raw ignore it.)
from jepa_quant.config import JEPAConfig, PriceEncoderConfig, DataConfig, TrainConfig
from jepa_quant.eval.diagnostics import load_checkpoint
from jepa_quant.training.trainer import build_components

cfg = JEPAConfig(
    price_encoder=PriceEncoderConfig(
        backend='transformer', n_features=6, context_length=64,
        d_model=768, n_heads=12, n_layers=8, latent_dim=256, freeze_backbone=True,
    ),
    data=DataConfig(
        data_dir=DATA_DIR, context_length=64, horizon=HORIZON, val_fraction=0.15, normalize=True,
    ),
    train=TrainConfig(batch_size=256, num_workers=2, device=DEVICE, seed=SEED),
)

trained = load_checkpoint(CHECKPOINT_PATH, cfg, device=DEVICE)
print(f'trained loaded from {CHECKPOINT_PATH}')

torch.manual_seed(SEED)
untrained = build_components(cfg)
for mod in (untrained.price_encoder, untrained.target_encoder, untrained.predictor, untrained.regularizer):
    mod.to(DEVICE).eval()
print('untrained baseline built (random init, same arch)')

In [ ]:
# == build encode_fns + the forecast head ==
# TimesFM weights download once and are cached by HF; the mean-pool encoder and the
# forecast head are built up front, the last-pool variant only in Test 4.
from jepa_quant.eval import (
    raw_feature_encode_fn, raw_sigma_encode_fn, timesfm_encode_fn, timesfm_forecast_fn,
)
from jepa_quant.eval.linear_probe import CLOSE_IDX

timesfm_mean = timesfm_encode_fn(TIMESFM_CHECKPOINT, DEVICE, variant='close', pool='mean')
timesfm_fcst = timesfm_forecast_fn(TIMESFM_CHECKPOINT, DEVICE, channel=CLOSE_IDX, horizon=HORIZON)

reps = {
    'raw_input':        raw_feature_encode_fn(),
    'untrained_zprice': untrained.price_encoder,
    'trained_zprice':   trained.price_encoder,
    'timesfm':          timesfm_mean,
}
ORDER = ['raw_input', 'untrained_zprice', 'trained_zprice', 'timesfm']
LABELS = {'raw_input': 'raw-input (baseline)', 'untrained_zprice': 'untrained z_price (FLOOR)',
          'trained_zprice': 'trained z_price', 'timesfm': 'TimesFM raw (CEILING)'}
print('reps:', ORDER)

## Test 1 — what does TimesFM actually output for L=64?

The encode path does `last_hidden_state.mean(dim=1)` assuming dim=1 is ~2 input patches (L=64, patch=32). If the HF wrapper instead pads the context up to the model's `context_length` (e.g. 512), dim=1 is dominated by pad positions and the mean-pool is corrupted. This prints the real shape so we stop guessing.

In [ ]:
# == Test 1: output shape instrumentation ==
import json
from jepa_quant.eval import instrument_timesfm_shapes

shapes = instrument_timesfm_shapes(TIMESFM_CHECKPOINT, cfg, channel=CLOSE_IDX,
                                   n_samples=8, device=DEVICE)
print(json.dumps(shapes, indent=2, default=float))
npos = shapes.get('n_positions')
print()
if npos is None:
    print("WARN: out has no last_hidden_state (see WARNING above) -> timesfm_encode_fn's pooling "
          "assumption is wrong; pick a hidden field from output_fields and update the encode_fn.")
elif npos <= 4:
    print(f"OK: n_positions={npos} ~ ceil(L/patch). mean-pool averages real patches.")
else:
    print(f"WARN: n_positions={npos} >> ~2 expected for L={shapes['context_length']}. The wrapper "
          f"likely padded to its context_length -> mean-pool is corrupted by pad positions. "
          f"Re-read Test 3/4 TimesFM numbers under pool='last'.")

## Test 2 — TimesFM forecast vs actual

TimesFM *used as intended*: its forecast-head point prediction for the next H steps, overlaid on the true target window (normalized close log-returns). If the forecast is flat or wildly off, that is direct visual evidence our differenced / per-sample-normalized input is out-of-distribution for a level-pretrained forecaster.

In [ ]:
# == Test 2: forecast-vs-actual overlays ==
from jepa_quant.eval import collect_timesfm_forecasts, plot_forecast_overlays

fc = collect_timesfm_forecasts(cfg, timesfm_fcst, n_samples=N_FORECAST_SAMPLES,
                               channel=CLOSE_IDX, split='val', device=DEVICE)
print('collected:', {k: tuple(v.shape) for k, v in fc.items() if hasattr(v, 'shape')})
plot_forecast_overlays(fc, n=N_FORECAST_SAMPLES, ncols=3)

## Test 3 — predicted vs actual (future_volatility)

The points behind nb04's R²s. Each representation's ridge probe (best α on the WIDE grid) predicts future volatility; we scatter predicted vs actual. A representation that "sees" volatility hugs the diagonal (positive R²); TimesFM's −0.134 should look like a structureless / negatively-sloped cloud next to raw-input's +0.333 diagonal.

In [ ]:
# == Test 3: predicted-vs-actual scatters ==
from jepa_quant.eval import probe_regression_predictions, plot_pred_vs_actual_grid

preds = [probe_regression_predictions(
            reps[n], cfg, target_kind='future_volatility',
            n_train_batches=N_TRAIN_BATCHES, n_val_batches=N_VAL_BATCHES,
            ridge_alphas=WIDE_ALPHAS, seed=SEED, device=DEVICE, source_label=LABELS[n])
         for n in ORDER]

print('=== future_volatility val R^2 (points plotted below) ===')
for p in preds:
    print(f"  {p['source_label']:<26} val R^2 {p['val_r2']:+.4f}   "
          f"train {p['train_r2']:+.4f}   best_a {p['best_alpha']:g}")
plot_pred_vs_actual_grid(preds, ncols=2)

## Test 4 — fix matrix (forward passes only)

Toggle one lever at a time and read future_volatility val R² (WIDE grid). The decisive contrast is **raw close-σ on RAW vs NORMALIZED context**: if σ-on-raw recovers ≈ +0.33 while σ-on-normalized ≈ 0, per-sample normalization is what erases the volatility-level signal — the input pipeline, not the backbone, is the lever.

In [ ]:
# == Test 4: fix experiments ==
from jepa_quant.eval import probe_regression_features

def vol_r2(enc, *, normalize_context=True, label='x'):
    return probe_regression_features(
        enc, cfg, target_kind='future_volatility',
        n_train_batches=N_TRAIN_BATCHES, n_val_batches=N_VAL_BATCHES,
        ridge_alphas=WIDE_ALPHAS, seed=SEED, device=DEVICE,
        source_label=label, normalize_context=normalize_context)

# Last-pool TimesFM (separate weight copy; relevant only if Test 1 flagged padding).
timesfm_last = timesfm_encode_fn(TIMESFM_CHECKPOINT, DEVICE, variant='close', pool='last')

experiments = {
    'raw_input        (NORM ctx)':  vol_r2(raw_feature_encode_fn(), normalize_context=True,  label='raw_norm'),
    'raw_input        (RAW  ctx)':  vol_r2(raw_feature_encode_fn(), normalize_context=False, label='raw_raw'),
    'raw close-sigma  (NORM ctx)':  vol_r2(raw_sigma_encode_fn(),   normalize_context=True,  label='sig_norm'),
    'raw close-sigma  (RAW  ctx)':  vol_r2(raw_sigma_encode_fn(),   normalize_context=False, label='sig_raw'),
    'timesfm mean     (NORM ctx)':  vol_r2(timesfm_mean, normalize_context=True,  label='tfm_mean_norm'),
    'timesfm last     (NORM ctx)':  vol_r2(timesfm_last, normalize_context=True,  label='tfm_last_norm'),
    'timesfm mean     (RAW  ctx)':  vol_r2(timesfm_mean, normalize_context=False, label='tfm_mean_raw'),
}

amin, amax = min(WIDE_ALPHAS), max(WIDE_ALPHAS)
print('=== fix experiments: future_volatility val R^2 (WIDE grid) ===')
for k, r in experiments.items():
    pin = '  [PINNED]' if r['best_alpha'] in (amin, amax) else ''
    print(f"  {k:<30} val R^2 {r['val_r2']:+.4f}   train {r['train_r2_at_best_alpha']:+.4f}   "
          f"best_a {r['best_alpha']:<8g} D={r['latent_dim']}{pin}")
print()
print('READ: raw close-sigma RAW >> NORM (and ~ raw_input NORM +0.33) => normalization removes the')
print('vol-level signal -> fix the INPUT PIPELINE (feed un-normalized ctx / a raw-sigma channel),')
print('not the backbone. [PINNED] best_alpha on a grid edge => widen WIDE_ALPHAS and re-run.')

## Summary & how to record

Fill the header **RESULT** and **DECISION + NEXT ACTION** from the four tests:
- Test 1: `n_positions` (was mean-pool over real patches or padded positions?).
- Test 2: did TimesFM's forecast track the target at all?
- Test 3: TimesFM vs raw-input vs floor val R² + the scatter shapes.
- Test 4: the raw-σ RAW-vs-NORM contrast (the normalization-is-the-lever test) and whether un-norm / last-pool lift TimesFM above the floor.

**Do NOT write a `research_log/` entry until the Colab numbers + figures are pasted back** (same rule as nb03c / nb04). This notebook is diagnostic; the project HOLD on the TimesFM pivot stands unless Test 4 shows a backbone can clear the +0.33 bar once the input pipeline is fixed.